# 05 · 数据管理与模型训练
使用模块四的真实 MuJoCo episode，完成质量检查、版本快照、按采样分组划分、官方 ACT 网络训练与闭环评估，以及 DINO-WM 的视觉预测和反事实动作推演。

DINOv2 预训练权重首次从 Meta 官方地址下载并转为 safetensors；后续复用本工作区的不可变权重。ACT 采用 LeRobot 官方网络，DINO-WM 使用官方因果 ViT 核心，并明确记录残差预测头、小型 RGB 解码器和示例分辨率等适配。示例训练结果不代表论文评测成绩。

In [ ]:
import json
import uuid

from examples._display import setup_notebook

PROJECT_ROOT, OUTPUT_DIR = setup_notebook("data_management_and_training")

from physrealx import ArtifactRef, PhysRealX, RunSpec
from physrealx.domain import TrainingSpec
from physrealx.integrations.datasets import (
    dataset_bundle,
    load_snapshot,
)

px = PhysRealX.open(PROJECT_ROOT / ".physrealx")
run_key = uuid.uuid4().hex[:12]


## 1. 仿真数据

In [ ]:
upstream = json.loads(
    (PROJECT_ROOT / "examples/simulation_and_data_generation/output/results.json").read_text()
)
collection = ArtifactRef.model_validate(upstream["episode_collection"])
evaluation_run = RunSpec.model_validate(upstream["evaluation_run"])


## 2. 数据接入

In [ ]:
operation = px.datasets.ingest(
    collection, method="episode_dataset", config={"resource_id": f"oven4.dataset.{run_key}"}
)
dataset_artifact = operation.output("dataset")
dataset = dataset_bundle(px.artifacts, dataset_artifact)
print(dict(operation.metrics))


## 3. 分组快照

In [ ]:
operation = px.datasets.snapshot(dataset_artifact, method="episode_split", config={"seed": 2026})
snapshot_ref = operation.output("snapshot")
snapshot, _, partitions = load_snapshot(px.artifacts, snapshot_ref)
train_ref = px.datasets.training_subset(
    snapshot_ref,
    resource_id=f"oven4.expert.train.{run_key}",
    required_tag="expert_demonstration",
)
print({name: len(values) for name, values in partitions.items()})


## 4. 策略训练

In [ ]:
import torch

from physrealx.integrations.act import ACTTrainer

device = "cuda" if torch.cuda.is_available() else "cpu"
model_ref = px.training.submit(
    TrainingSpec(
        resource_id=f"oven4.act.{run_key}",
        version="1.0.0",
        dataset=train_ref,
        trainer=ACTTrainer.descriptor.bind(
            config={
                "steps": 300,
                "device": device,
                "seed": 2026,
                "observation_signals": [
                    "joint.position",
                    "joint.velocity",
                    "joint.oven.e_door_197.joint.position",
                    "joint.oven.e_door_197.joint.velocity",
                ],
            }
        ),
    )
)
model = px.training.get_model(model_ref)
print(device, model.metrics)


## 5. 闭环评测

In [ ]:
evaluation_ref = px.evaluations.submit(
    resource_id=f"oven4.act.validation.{run_key}",
    version="1.0.0",
    model=model_ref,
    run=evaluation_run,
    episode_count=5,
    base_seed=91000,
)
evaluation = px.evaluations.get_report(evaluation_ref)
# Record the initial policy honestly; final policy acceptance follows feedback in module six.
assert evaluation.metrics["episode_count"] == 5
print("独立闭环成功率:", evaluation.success_rate)


## 6. 世界模型训练

In [ ]:
encoder_ref = px.world_models.prepare_encoder(
    cache=PROJECT_ROOT / "examples/data_management_and_training/input/dinov2-encoder.json"
)
operation = px.world_models.train(
    snapshot_ref,
    encoder_ref,
    method="dino_wm",
    config={
        "device": device,
        "steps": 1800,
        "history": 1,
        "training_horizon": 100,
        "evaluation_horizon": 100,
        "validation_interval": 200,
        "state_loss_weight": 5,
        "state_signals": [
            "joint.position",
            "joint.velocity",
            "joint.oven.e_door_197.joint.position",
            "joint.oven.e_door_197.joint.velocity",
        ],
    },
)
for metric in ("normalized_state_rmse", "latent_mse", "rgb_mse"):
    assert operation.metrics["test_" + metric] < operation.metrics["test_persistence_" + metric], (
        metric
    )
world_model_ref = operation.output("model")
world_report_ref = operation.output("training_report")
print(dict(operation.metrics))


## 7. 未来预测

In [ ]:
import numpy as np

from physrealx.integrations._artifacts import read_array
from physrealx.integrations.world_model import WorldModelActions, WorldModelContext

test_episode = next(episode for episode in partitions["test"] if episode.success)
test_summary = next(row for row in dataset.episodes if row.episode_id == test_episode.episode_id)
start_index, horizon = 0, 12
context_ref = px.artifacts.put_json(
    WorldModelContext(episode=test_summary, start_index=start_index).model_dump(
        mode="json", by_alias=True
    )
)
future_actions = [
    step.action.inline for step in test_episode.steps[start_index : start_index + horizon]
]
interval = test_episode.steps[1].timestamp_seconds - test_episode.steps[0].timestamp_seconds
actions_ref = px.artifacts.put_json(
    WorldModelActions(values=future_actions, interval_seconds=interval).model_dump(
        mode="json", by_alias=True
    )
)
operation = px.world_models.rollout(
    world_model_ref, context_ref, actions_ref, method="dino_wm", config={"device": device}
)
rollout = json.loads(px.artifacts.get_bytes(operation.output("rollout")))
predicted = read_array(px.artifacts, ArtifactRef.model_validate(rollout["states"]))
future_rgb = read_array(px.artifacts, ArtifactRef.model_validate(rollout["rgb"]))
truth = np.array(
    [
        step.observations["joint.position"].inline[0]
        for step in test_episode.steps[start_index + 1 : start_index + horizon + 1]
    ]
)
position_rmse_m = float(np.sqrt(np.mean((predicted[:, 0] - truth) ** 2)))
assert np.isfinite(predicted).all() and future_rgb.shape[0] == horizon
print("未参与训练 episode 的预测位置 RMSE (m):", position_rmse_m)


## 8. 反事实对照

In [ ]:
import matplotlib.pyplot as plt

from physrealx import MethodCapability

changed_actions = [[0.0] for _ in range(100)]
changed_ref = px.artifacts.put_json(
    WorldModelActions(values=changed_actions, interval_seconds=interval)
)
operation = px.world_models.rollout(
    world_model_ref, context_ref, changed_ref, method="dino_wm", config={"device": device}
)
counterfactual_ref = operation.output("rollout")
changed = json.loads(px.artifacts.get_bytes(counterfactual_ref))
counterfactual = read_array(px.artifacts, ArtifactRef.model_validate(changed["states"]))
# Reuse the exact scene, task, backend and seed. Only future robot actions change.
counter_run = px.runs.submit(
    RunSpec(
        scene=test_episode.scene,
        task=test_episode.task,
        backend=test_episode.backend,
        policy=px.methods.bind(
            MethodCapability.POLICY_INFERENCE,
            "joint_sequence",
            config={"commands": changed_actions},
        ),
        episode_count=1,
        base_seed=test_episode.seed,
        stop_on_success=False,
        dataset_id=f"oven4.counterfactual.{run_key}",
        dataset_version="1.0.0",
        record_signals=evaluation_run.record_signals,
        tags=("counterfactual_truth", test_episode.episode_id),
    )
)
counter_episode = px.runs.episodes(counter_run)[0]
counter_evaluation_ref = px.world_models.evaluate_rollout(
    counterfactual_ref, counter_episode, start_index=0
)
counter_evaluation = json.loads(px.artifacts.get_bytes(counter_evaluation_ref))
state_names = tuple(changed["state_signals"])
counter_truth = read_array(
    px.artifacts, ArtifactRef.model_validate(counter_evaluation["reference_states"])
)
door_counterfactual = counter_evaluation["signals"][
    "joint.oven.e_door_197.joint.position"
]
assert door_counterfactual["status"] == ["BEATS_PERSISTENCE"]
print(
    "五秒反事实门角 RMSE / persistence (rad):",
    door_counterfactual["rmse"][0],
    "/",
    door_counterfactual["persistence_rmse"][0],
)


## 9. 五秒未来推演
世界模型配置 history=1，只使用起点的一帧 RGB 和同步状态。下方生成 100 步、共五秒的动作条件未来图像与状态；超出已记录轨迹的部分用于展示推演接口，不能计入有真值的预测精度。


In [ ]:
five_second_steps = round(5.0 / interval)
assert len(test_episode.steps) > five_second_steps
continuation = [s.action.inline for s in test_episode.steps[:five_second_steps]]
long_actions = px.artifacts.put_json(
    WorldModelActions(values=continuation, interval_seconds=interval)
)
operation = px.world_models.rollout(
    world_model_ref, context_ref, long_actions, method="dino_wm", config={"device": device}
)
long_prediction_ref = operation.output("rollout")
prediction = json.loads(px.artifacts.get_bytes(long_prediction_ref))
long_states = read_array(px.artifacts, ArtifactRef.model_validate(prediction["states"]))
long_evaluation_ref = px.world_models.evaluate_rollout(
    long_prediction_ref, test_episode, start_index=0
)
long_evaluation = json.loads(px.artifacts.get_bytes(long_evaluation_ref))
long_truth = read_array(
    px.artifacts, ArtifactRef.model_validate(long_evaluation["reference_states"])
)
position_signals = (
    "joint.position",
    "joint.oven.e_door_197.joint.position",
)
assert all(
    long_evaluation["signals"][name]["status"] == ["BEATS_PERSISTENCE"]
    for name in position_signals
)
robot_column = long_evaluation["signals"][position_signals[0]]["column_start"]
door_column = long_evaluation["signals"][position_signals[1]]["column_start"]
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for axis, index, name in zip(
    axes, (robot_column, door_column), ("Robot pusher", "Passive oven door")
):
    axis.plot(long_truth[:, index], label="MuJoCo truth")
    axis.plot(long_states[:, index], label="Predicted")
    axis.plot(counterfactual[:, index], label="Counterfactual prediction")
    axis.plot(counter_truth[:, index], linestyle="--", label="Counterfactual truth")
    axis.set_title(name)
    axis.set_ylabel("Position / " + ("m" if index == robot_column else "rad"))
    axis.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "world-model.png", dpi=140)
plt.show()
print(
    "五秒事实位置 RMSE / persistence:",
    {
        name: (
            long_evaluation["signals"][name]["rmse"][0],
            long_evaluation["signals"][name]["persistence_rmse"][0],
        )
        for name in position_signals
    },
)


## 10. 结果导出

In [ ]:
results = {
    "model": model_ref.model_dump(mode="json"),
    "world_model": world_model_ref.model_dump(mode="json"),
    "train_dataset": train_ref.model_dump(mode="json"),
    "snapshot": snapshot_ref.model_dump(mode="json"),
    "dataset": dataset_artifact.model_dump(mode="json"),
    "evaluation_run": evaluation_run.model_dump(mode="json"),
    "world_model_training": world_report_ref.model_dump(mode="json"),
    "long_prediction": long_prediction_ref.model_dump(mode="json"),
    "long_prediction_evaluation": long_evaluation_ref.model_dump(mode="json"),
    "counterfactual_prediction": counterfactual_ref.model_dump(mode="json"),
    "counterfactual_evaluation": counter_evaluation_ref.model_dump(mode="json"),
    "counterfactual_dataset": counter_run.result.dataset.model_dump(mode="json"),
}
results_ref = px.artifacts.export(
    OUTPUT_DIR,
    results,
    files={"world-model.json": world_model_ref, "world-model-training.json": world_report_ref},
)
print("结果索引:", OUTPUT_DIR / "results.json")
